In [2]:
import pandas as pd
import duckdb

## Parte 1: Exploração inicial dos dados

In [ ]:
# carregando dados
df = pd.read_csv("../data/DOHMH_New_York_City_Restaurant_Inspection_Results.csv")
df.head() # por padrão só exibe as 5 primeiras linhas 

### 1. número de linhas e colunas;
 

In [ ]:
print(df.shape)  # exibe o número de linhas e colunas

### 2. tipos dos atributos;

In [ ]:
df.dtypes #tipo das colunas

###  3. porcentagem de valores nulos por coluna;

In [ ]:
nulos = (df.isnull().sum()/len(df))*100
nulos.sort_values(ascending=False)

### 4. número de valores distintos por coluna;

In [ ]:
df.nunique() #valores dististos 

### 5. exemplos de valores frequentes para colunas categóricas;

In [ ]:
categoricas = df.select_dtypes(include='object').columns

for coluna in categoricas: #a cada coluna aparece os 5 valores que mais apareceram
    print(f"\n{coluna}")
    print(df[coluna].value_counts().head(5)) 

###  6. estatísticas simples para colunas numéricas e temporais.

In [ ]:
df['INSPECTION DATE'] = pd.to_datetime(df['INSPECTION DATE'], errors='coerce')

print("Data mínima:", df['INSPECTION DATE'].min())
print("Data máxima:", df['INSPECTION DATE'].max())

## Parte 2: Escrita manual de regras candidatas


FD1: Cada restaurante identificado por um CAMIS possui um único nome comercial (DBA). ```CAMIS → DBA```

FD2: Cada CAMIS deve ter apenas 1 telefone. ```CAMIS → PHONE```

CFD1: GRADE 'A' precisa ter SCORE abaixo de 13. ```GRADE = 'A' ⇒ SCORE ≤ 13```

CFD2: BORO 'Manhattan' ZIPCODE comeca com 10. ```BORO = 'Manhattan' ⇒ ZIPCODE LIKE '10%'```

DC1: Nenhum restaurante pode ter SCORE negativo. ```¬(SCORE < 0)```

DC2: Não pode existir uma inspeção com data de inspeção posterior à data de registro.```¬(INSPECTION DATE > RECORD DATE)```

### Restrições de dados usando SQL

In [ ]:
import duckdb

duckdb.register("restaurants", df)

In [ ]:
duckdb.sql( """SELECT COUNT(*)
FROM (
    SELECT CAMIS
    FROM restaurants
    GROUP BY CAMIS
    HAVING COUNT(DISTINCT DBA) > 1
);""")



In [ ]:
duckdb.sql( """SELECT COUNT(*)
FROM (
    SELECT CAMIS
    FROM restaurants
    GROUP BY CAMIS
    HAVING COUNT(DISTINCT PHONE) > 1
);""")

In [25]:
duckdb.sql( """SELECT COUNT(*)
FROM restaurants
WHERE GRADE = 'A'
  AND SCORE > 13;""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│           13 │
└──────────────┘

In [26]:
duckdb.sql( """SELECT COUNT(*)
FROM restaurants
WHERE BORO = 'Manhattan'
  AND CAST(ZIPCODE AS VARCHAR) NOT LIKE '10%';""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│            7 │
└──────────────┘

In [27]:
duckdb.sql( """SELECT COUNT(*)
FROM restaurants
WHERE SCORE < 0; """)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘

In [ ]:
df["INSPECTION DATE"] = pd.to_datetime(df["INSPECTION DATE"])
df["RECORD DATE"] = pd.to_datetime(df["RECORD DATE"]) ##cnvertendo as datas

In [30]:
duckdb.register("restaurants", df)

In [31]:
duckdb.sql( """SELECT COUNT(*)
FROM restaurants
WHERE "INSPECTION DATE" > "RECORD DATE"; """)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘